# BNPL Capstone -  Data Collection

This notebook pulls:
1. **CFPB Complaint Database** — real complaints about Affirm, Klarna, Afterpay, Zip, Sezzle
2. **FRED Economic Data** — macro indicators (delinquency rate, savings rate, etc.)
3. **Lending Club** — labeled loan default data for ML modeling


## Mount Drive & Install Packages

In [ ]:
# mounting google drive and creating the folder structure needed to store raw and processed data.
# we expect the drive to mount successfully and all project folders to be created without errors.from google.colab import drive
drive.mount('/content/drive')

import io
import os
import pandas as pd
import numpy as np

BASE = '/content/drive/MyDrive/BNPL_Capstone'
for folder in ['data/raw','data/processed','output/figures','output/models']:
    os.makedirs(f'{BASE}/{folder}', exist_ok=True)
print('Project folders ready:', BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folders ready: /content/drive/MyDrive/BNPL_Capstone


In [ ]:
# installing fredapi, tqdm, and pyarrow which are needed for fetching macro data and saving parquet files.
# we expect all three packages to install without errors before the data collection steps begin.

!pip install -q fredapi tqdm pyarrow
print('extra packages installed')

extra packages installed


## 1 · CFPB Consumer Complaint Database

The government has a website where people go to complain when a company treats them badly. It is called [Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/#get-the-data). This files downloads every complaint people filed against Affirm, Klarna, Afterpay, Zip, Sezzle, and PayPal.

This dataset has  collection of things like:



*   When did they complain?
*   What state do they live in?
*   What was the problem?
* Did the company respond on time?


This tells : where are people most angry and why?

In [ ]:
# loading the cfpb complaints csv in chunks and filtering for bnpl companies: affirm, klarna, afterpay, zip, sezzle, and paypal.
# we expect approximately 50,000 complaint records to be retained after filtering and saved to drive.

BNPL_COMPANIES = ['Affirm', 'Klarna', 'Afterpay', 'Zip', 'Sezzle', 'PayPal']

# Only keep columns needed for this project
needed_cols = ['Date received', 'Company', 'Product', 'State']

bnpl_keywords = ['affirm', 'klarna', 'afterpay', 'zip', 'sezzle', 'paypal']

chunks = []
chunk_size = 200000

for chunk in pd.read_csv(
    f'{BASE}/data/raw/complaints.csv',
    usecols=needed_cols,
    chunksize=chunk_size,
    low_memory=False
):
    # Standardize column names inside each chunk
    chunk.columns = (
        chunk.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
    )

    # Standardize company values before filtering
    chunk['company'] = chunk['company'].astype(str).str.strip().str.lower()

    # Keep rows where company name contains any BNPL keyword
    pattern = '|'.join(bnpl_keywords)
    chunk = chunk[chunk['company'].str.contains(pattern, na=False)]

    if not chunk.empty:
        chunks.append(chunk)

# Handle case where no rows matched
if len(chunks) == 0:
    raise ValueError(
        "No BNPL complaint records were found. Check the company names in the CFPB file."
    )

# Combine filtered chunks
complaints = pd.concat(chunks, ignore_index=True)

# Parse dates
complaints['date_received'] = pd.to_datetime(complaints['date_received'], errors='coerce')
complaints['year'] = complaints['date_received'].dt.year
complaints['month'] = complaints['date_received'].dt.month

# Save cleaned version
complaints.to_csv(f'{BASE}/data/raw/cfpb_complaints.csv', index=False)

print(f'Saved {len(complaints):,} total complaints to Drive')
complaints.head()

Saved 50,681 total complaints to Drive


,date_received,product,company,state,year,month
0,2019-08-11,Credit card or prepaid card,"paypal holdings, inc",MO,2019,8
1,2024-01-15,Debt collection,"affirm holdings, inc",NC,2024,1
2,2024-01-31,"Money transfer, virtual currency, or money ser...","paypal holdings, inc",CA,2024,1
3,2022-12-03,"Money transfer, virtual currency, or money ser...","paypal holdings, inc",NaN,2022,12
4,2020-03-05,"Money transfer, virtual currency, or money ser...","paypal holdings, inc",NY,2020,3


## 2 · FRED Macro Indicators
> Get an **API** at [Federal Reserve of ST.Louis](https://fred.stlouisfed.org/docs/api/fred/v2/index.html)

FRED (Government economic data)
The Federal Reserve (the US government's money people) tracks giant numbers about the whole country's finances. This dataset has a colleciton of things like:

* How many Americans are late on their credit cards right now?
* How much money are people saving?
* How high are interest rates?
* How many people are unemployed?

This tells: when the economy is bad, do BNPL problems get worse?

In [ ]:
# fetching eight macroeconomic time series from the fred api covering delinquency rates, savings, inflation, and unemployment from 2018 onward.
# we expect all eight series to download successfully and be saved as a single long-format csv with 974 rows.
from fredapi import Fred
from tqdm import tqdm
import time

FRED_API_KEY = 'f0073dd2a90a7405dea3f88d092ff267'

fred = Fred(api_key=FRED_API_KEY)

FRED_SERIES = {
        'DRCCLACBS':      'Credit Card Delinquency Rate',
        'DRSFRMACBS':     'Consumer Loan Delinquency Rate',
        'PSAVERT':        'Personal Savings Rate',
        'CCLACBW027SBOG': 'Consumer Credit: Revolving',
        'TERMCBCCALLNS':  'Credit Card Interest Rate',
        'UMCSENT':        'Consumer Sentiment Index',
        'UNRATE':         'Unemployment Rate',
        'CPIAUCSL':       'CPI (Inflation Proxy)',
    }
frames = []
for sid, name in tqdm(FRED_SERIES.items(), desc='Fetching FRED'):
        try:
            s = fred.get_series(sid, observation_start='2018-01-01')
            df = s.reset_index()
            df.columns = ['date', 'value']
            df['indicator'] = name
            df['series_id'] = sid
            frames.append(df)
            time.sleep(0.2)
        except Exception as e:
            print(f'  {sid}: {e}')
macro = pd.concat(frames, ignore_index=True)
macro.to_csv(f'{BASE}/data/raw/fred_macro.csv', index=False)
print(f'Saved {len(macro):,} rows of macro data')
macro.pivot(index='date', columns='indicator', values='value').tail()

Fetching FRED: 100%|██████████| 8/8 [00:03<00:00,  2.55it/s]

Saved 974 rows of macro data


indicator,CPI (Inflation Proxy),Consumer Credit: Revolving,Consumer Loan Delinquency Rate,Consumer Sentiment Index,Credit Card Delinquency Rate,Credit Card Interest Rate,Personal Savings Rate,Unemployment Rate
date,,,,,,,,
2026-02-01,327.46,NaN,NaN,NaN,NaN,NaN,NaN,4.4
2026-02-04,NaN,1069.7639,NaN,NaN,NaN,NaN,NaN,NaN
2026-02-11,NaN,1070.3662,NaN,NaN,NaN,NaN,NaN,NaN
2026-02-18,NaN,1071.6423,NaN,NaN,NaN,NaN,NaN,NaN
2026-02-25,NaN,1068.7495,NaN,NaN,NaN,NaN,NaN,NaN


## 3 · Lending Club Loan Data

> . [Download  dataset from Kaggle -All Lending Club loan data](https://www.kaggle.com/datasets/wordsforthewise/lending-club?select=accepted_2007_to_2018Q4.csv.gz) then use `accepted_2007_to_2018Q4.csv`

Lending Club loans
This is a dataset from Kaggle  with millions of real loans — and most importantly, it tells who paid back their loan and who didn't. This is the training data for the AI model.
This dataset has a collection of  things like:

* Person's credit score
* Their income
* How much debt they already have
* Did they default (not pay back)?


In [ ]:
# loading the lending club dataset, creating a binary default target from loan_status, and cleaning the interest rate and date columns.
# we expect 1,348,099 loans to be retained with a default rate near 20% after filtering to paid and defaulted statuses only.

KEY_COLS = [
    'loan_amnt','funded_amnt','term','int_rate','installment',
    'grade','sub_grade','emp_length','home_ownership','annual_inc',
    'verification_status','loan_status','purpose','dti',
    'delinq_2yrs','fico_range_low','fico_range_high',
    'open_acc','pub_rec','revol_bal','revol_util',
    'total_acc','mort_acc','pub_rec_bankruptcies','addr_state','issue_d'
]

DEFAULT_STATUSES = {'Charged Off','Default',
                    'Does not meet the credit policy. Status:Charged Off'}
REPAID_STATUSES  = {'Fully Paid',
                    'Does not meet the credit policy. Status:Fully Paid'}

loans = pd.read_csv(
    f'{BASE}/data/raw/accepted_2007_to_2018Q4.csv',
    usecols=KEY_COLS,
    low_memory=False
)

loans['default'] = loans['loan_status'].map(
    {**{s:1 for s in DEFAULT_STATUSES}, **{s:0 for s in REPAID_STATUSES}})
loans = loans.dropna(subset=['default'])
loans['default'] = loans['default'].astype(int)

for col in ['int_rate','revol_util']:
    if col in loans.columns:
        loans[col] = pd.to_numeric(loans[col].astype(str).str.replace('%',''), errors='coerce')

loans['issue_date'] = pd.to_datetime(loans['issue_d'], format='%b-%Y', errors='coerce')
loans['issue_year'] = loans['issue_date'].dt.year

loans.to_parquet(f'{BASE}/data/raw/lending_club_raw.parquet', index=False)
print(f'Saved {len(loans):,} loans | {loans["default"].mean()*100:.1f}% default rate')
loans.head()
loans.head()

Saved 1,348,099 loans | 20.0% default rate


,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,mort_acc,pub_rec_bankruptcies,default,issue_date,issue_year
0,3600.0,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,55000.0,...,7.0,0.0,2765.0,29.7,13.0,1.0,0.0,0,2015-12-01,2015
1,24700.0,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,65000.0,...,22.0,0.0,21470.0,19.2,38.0,4.0,0.0,0,2015-12-01,2015
2,20000.0,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,63000.0,...,6.0,0.0,7869.0,56.2,18.0,5.0,0.0,0,2015-12-01,2015
4,10400.0,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,104433.0,...,12.0,0.0,21929.0,64.5,35.0,6.0,0.0,0,2015-12-01,2015
5,11950.0,11950.0,36 months,13.44,405.18,C,C3,4 years,RENT,34000.0,...,5.0,0.0,8822.0,68.4,6.0,0.0,0.0,0,2015-12-01,2015


## 4. Build the Combined Modeling Dataset

### 4.1 Load cleaned source datasets

The three source datasets have already been collected and cleaned separately. The next step is to combine them into a single loan-level dataset that can be used for later analysis and modeling.

The Lending Club dataset remains the core dataset because it contains the borrower characteristics and repayment outcomes needed for prediction. The FRED data and CFPB complaint data are added as contextual features. Since those external datasets are not available at the borrower level, they are merged using time-based indicators, and where possible, state-level complaint counts are also included.

In [ ]:
# reloading the three cleaned source datasets from drive to begin building the combined modeling dataset.
# we expect loans with 1,348,099 rows, complaints with 50,681 rows, and macro with 974 rows.

loans = pd.read_parquet(f'{BASE}/data/raw/lending_club_raw.parquet')
complaints = pd.read_csv(f'{BASE}/data/raw/cfpb_complaints.csv', low_memory=False)
macro = pd.read_csv(f'{BASE}/data/raw/fred_macro.csv', low_memory=False)

print("Source dataset shapes:")
print("  Loans      :", loans.shape)
print("  Complaints :", complaints.shape)
print("  Macro      :", macro.shape)

Source dataset shapes:
  Loans      : (1348099, 29)
  Complaints : (50681, 6)
  Macro      : (974, 4)


### 4.2 Create common time keys

The three datasets do not share a borrower identifier, so they cannot be joined directly. Instead, the integration is anchored on time. For Lending Club, the relevant date is the loan issue date. For CFPB complaints, it is the complaint receipt date. For FRED, it is the observation date for each macroeconomic series.

A monthly key is created for each dataset so that external conditions can be aligned with the month in which a loan was issued.

In [ ]:
## standardizing date formats across all three datasets and creating a shared monthly key so they can be merged on time.
# we expect each dataset to have a new issue_month column in yyyy-mm format with no missing date values remaining.

# Loans
loans = loans.copy()
loans['issue_date'] = pd.to_datetime(loans['issue_date'], errors='coerce')
loans = loans.dropna(subset=['issue_date']).copy()
loans['issue_month'] = loans['issue_date'].dt.to_period('M').astype(str)

# Complaints
complaints = complaints.copy()
complaints['date_received'] = pd.to_datetime(complaints['date_received'], errors='coerce')
complaints = complaints.dropna(subset=['date_received']).copy()
complaints['issue_month'] = complaints['date_received'].dt.to_period('M').astype(str)

# Macro
macro = macro.copy()
macro['date'] = pd.to_datetime(macro['date'], errors='coerce')
macro = macro.dropna(subset=['date']).copy()
macro['issue_month'] = macro['date'].dt.to_period('M').astype(str)

# Standardize state abbreviations where relevant
if 'addr_state' in loans.columns:
    loans['addr_state'] = loans['addr_state'].astype(str).str.strip().str.upper()

if 'state' in complaints.columns:
    complaints['state'] = complaints['state'].astype(str).str.strip().str.upper()

print("Monthly time keys created.")

Monthly time keys created.


### 4.3 Reshape the macroeconomic data

The FRED data are currently in long format, where each row represents one indicator on one date. The macroeconomic dataset is pivoted into a wide monthly format before merging.

In [ ]:
# pivoting the fred data from long format to wide format so each month becomes one row with one column per indicator.
# we expect a macro monthly table with 98 rows and 9 columns, with any small gaps filled using forward and backward fill.

macro_monthly = (
    macro.pivot_table(
        index='issue_month',
        columns='indicator',
        values='value',
        aggfunc='mean'
    )
    .reset_index()
)

# Clean column names
macro_monthly.columns.name = None
macro_monthly.columns = [
    str(col)
      .lower()
      .replace(' ', '_')
      .replace('(', '')
      .replace(')', '')
      .replace(':', '')
      .replace('-', '_')
    for col in macro_monthly.columns
]

# Sort by month and fill any small gaps in the macro series
macro_monthly = macro_monthly.sort_values('issue_month').copy()
macro_value_cols = [c for c in macro_monthly.columns if c != 'issue_month']
macro_monthly[macro_value_cols] = macro_monthly[macro_value_cols].ffill().bfill()

print("Macro monthly shape:", macro_monthly.shape)
macro_monthly.head()

Macro monthly shape: (98, 9)


,issue_month,cpi_inflation_proxy,consumer_credit_revolving,consumer_loan_delinquency_rate,consumer_sentiment_index,credit_card_delinquency_rate,credit_card_interest_rate,personal_savings_rate,unemployment_rate
0,2018-01,248.859,768.535920,3.48,95.7,2.5,13.63,5.7,4.0
1,2018-02,249.529,769.267350,3.48,99.7,2.5,13.63,5.8,4.1
2,2018-03,249.577,770.982625,3.48,101.4,2.5,13.63,5.9,4.0
3,2018-04,250.227,786.734550,3.22,98.8,2.5,13.63,6.0,4.0
4,2018-05,250.792,789.705940,3.22,98.0,2.5,14.14,6.0,3.8


### 4.4 Construct complaint-based contextual features

The CFPB dataset does not contain loan-level observations, so it is used to create contextual indicators instead. Two complaint features are constructed here: total complaint volume by month and complaint volume by month and state. These features are intended to capture broader patterns of credit stress and complaint activity in the lending environment.

In [ ]:
# aggregating cfpb complaint counts at the national monthly level and at the state-month level to use as contextual features.
# we expect 156 rows in the monthly aggregate and 5,644 rows in the state-month aggregate.

# National complaint volume by month
complaints_monthly = (
    complaints.groupby('issue_month')
    .size()
    .reset_index(name='cfpb_complaint_count')
)

# State-level complaint volume by month
if 'state' in complaints.columns and 'addr_state' in loans.columns:
    complaints_state_month = (
        complaints.groupby(['issue_month', 'state'])
        .size()
        .reset_index(name='cfpb_complaint_count_state')
        .rename(columns={'state': 'addr_state'})
    )
else:
    complaints_state_month = None

print("Complaint aggregates:")
print("  Monthly complaints      :", complaints_monthly.shape)
if complaints_state_month is not None:
    print("  State-month complaints  :", complaints_state_month.shape)

Complaint aggregates:
  Monthly complaints      : (156, 2)
  State-month complaints  : (5644, 3)


### 4.5 Merge the external context into the loan dataset

The Lending Club dataset remains the base table because each row represents one loan and includes the target variable for prediction. The monthly macroeconomic features and complaint indicators are merged into the loan data using the issue month, and state-level complaint features are also added where available.

In [ ]:
# merging the macroeconomic features and complaint counts into the loan dataset using the shared monthly time key.
# we expect the combined dataset to have 1,348,099 rows and 40 columns after all three merges complete.

model_df = loans.copy()

# Add macroeconomic context
model_df = model_df.merge(
    macro_monthly,
    on='issue_month',
    how='left'
)

# Add national complaint volume
model_df = model_df.merge(
    complaints_monthly,
    on='issue_month',
    how='left'
)

# Add state-month complaint volume if available
if complaints_state_month is not None:
    model_df = model_df.merge(
        complaints_state_month,
        on=['issue_month', 'addr_state'],
        how='left'
    )

print("Combined dataset shape after merges:", model_df.shape)

Combined dataset shape after merges: (1348099, 40)


### 4.6 Final cleanup of merged features

Complaint counts are filled with zero where no matching complaints are observed. For macroeconomic variables, missing values are carried forward or backward because those series represent continuous time-based indicators rather than event counts. Duplicate rows are removed to keep the final dataset clean.

In [ ]:
# filling missing complaint counts with zero and forward-filling any remaining gaps in the macroeconomic columns.
# we expect the final dataset to have 1,348,099 rows with no missing values in the merged feature columns.

complaint_cols = [
    'cfpb_complaint_count',
    'cfpb_complaint_count_state'
]

for col in complaint_cols:
    if col in model_df.columns:
        model_df[col] = model_df[col].fillna(0)

# Macro variables: fill remaining missing values using nearby observations
macro_cols = [c for c in macro_monthly.columns if c != 'issue_month']

model_df = model_df.sort_values('issue_date').copy()

for col in macro_cols:
    if col in model_df.columns:
        model_df[col] = model_df[col].ffill().bfill()

# Remove duplicates if any were created during merging
model_df = model_df.drop_duplicates().copy()

print("Final dataset shape:", model_df.shape)

Final dataset shape: (1348099, 40)


### 4.7 Save the final modeling dataset

The final output of this section is a single loan-level dataset that combines borrower characteristics, repayment outcomes, macroeconomic indicators, and complaint-based features.

In [ ]:
# saving the combined loan-level dataset with borrower, macroeconomic, and complaint features as both parquet and csv.
# we expect both files to save successfully to the processed data folder and the dataset shape to confirm 1,348,099 rows and 40 columns.

model_df.to_parquet(f'{BASE}/data/processed/bnpl_modeling_dataset.parquet', index=False)
model_df.to_csv(f'{BASE}/data/processed/bnpl_modeling_dataset.csv', index=False)

print("Saved combined modeling dataset:")
print(f"  {BASE}/data/processed/bnpl_modeling_dataset.parquet")
print(f"  {BASE}/data/processed/bnpl_modeling_dataset.csv")

# Review final structure
print("Final columns:")
print(model_df.columns.tolist())

print("\nPreview:")
model_df.head()

# export the merged dataset
OUTPUT_PATH = f'{BASE}/data/processed'

# Save dataset
model_df.to_csv(f'{OUTPUT_PATH}/bnpl_modeling_dataset.csv', index=False)
model_df.to_parquet(f'{OUTPUT_PATH}/bnpl_modeling_dataset.parquet', index=False)

print("Merged dataset successfully saved.")
print("CSV:", f'{OUTPUT_PATH}/bnpl_modeling_dataset.csv')
print("Parquet:", f'{OUTPUT_PATH}/bnpl_modeling_dataset.parquet')

print("\nDataset shape:", model_df.shape)

Saved combined modeling dataset:
  /content/drive/MyDrive/BNPL_Capstone/data/processed/bnpl_modeling_dataset.parquet
  /content/drive/MyDrive/BNPL_Capstone/data/processed/bnpl_modeling_dataset.csv
Final columns:
['loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies', 'default', 'issue_date', 'issue_year', 'issue_month', 'cpi_inflation_proxy', 'consumer_credit_revolving', 'consumer_loan_delinquency_rate', 'consumer_sentiment_index', 'credit_card_delinquency_rate', 'credit_card_interest_rate', 'personal_savings_rate', 'unemployment_rate', 'cfpb_complaint_count', 'cfpb_complaint_count_state']

Preview:
Merged dataset successfully saved.
CSV: /content/drive/MyDrive/BNPL_Capstone/data/pro